# MEM-76 · Выбор модели эмбеддингов: multilingual-e5 или BGE-M3

1. **Среда выполнения → Сменить среду → GPU (T4)**.
2. **Среда выполнения → Выполнить все** (≈ 30–35 мин: скачивание и распознавание 91 ролика — самое долгое).
3. В конце скачается `embeddings_check_results.zip` — пришли его в чат.

Что делает ноутбук:
1. Скачивает аудио всех роликов коллекции и распознаёт речь faster-whisper large-v3 (выбрана в MEM-16).
2. Режет транскрипт на перекрывающиеся фрагменты; подпись ролика — отдельный фрагмент.
3. Строит векторы фрагментов двумя моделями и ищет по косинусной близости в памяти, без БД. Оценка ролика — лучший из его фрагментов.
4. Пробует реранкер bge-reranker-v2-m3 поверх каждой модели: пересортировывает 50 лучших фрагментов (и меряет его скорость на CPU).
5. Прогоняет контрольные запросы MEM-17 и считает Recall@1/5/10, MRR, скорость и размер вектора. Для сравнения — простой поиск по словам (BM25).

Поиск идёт по всей коллекции из 91 ролика, а не только по тем, к которым есть запросы, — остальные ролики служат «шумом», как у настоящего пользователя.

In [ ]:
!pip -q install faster-whisper yt-dlp sentence-transformers rank_bm25 nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU нет — будет очень медленно"
# faster-whisper (CTranslate2) собран под CUDA 12, а в Colab может стоять CUDA 13 — подгружаем библиотеки CUDA 12 вручную
import ctypes
import glob

import nvidia.cublas
import nvidia.cudnn

for lib_dir in [nvidia.cublas.__path__[0] + "/lib", nvidia.cudnn.__path__[0] + "/lib"]:
    for so in sorted(glob.glob(lib_dir + "/*.so*"), key=lambda p: "Lt" not in p):
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pass
import torch  # noqa: E402

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# Коллекция (data/collection/passport.csv) и контрольные запросы (data/queries/control_queries.csv)
CLIPS = [
    {
        "n": 1,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@wolframoviy02/video/7693220621068684562",
        "caption": "История про Therac-25 #история #программист #мемы #программирование #programming  Therac-25 (с англ. — «Терак-25») — аппарат лучевой терапии, медицинский ускоритель, созданный канадской государственно",
        "speech": "да",
    },
    {
        "n": 2,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@ilnitskaja1/video/7492893285204249878",
        "caption": "Было…..тяжело 😳 #ягодичныймостик #ягодицы #мост #техника #ягодичныймост250кг #моствтренажере #зал #пп #упражнения #gymgirlsoftiktok ",
        "speech": "да",
    },
    {
        "n": 3,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@prof.zhukpuk/video/7696248119398386976",
        "caption": "я не знаю какие хештеги тут ставить, делитесь ссылкой, чтобы продвигать профэссора за базированное мнение",
        "speech": "да",
    },
    {
        "n": 4,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@ttbynina/video/7687297499735805202",
        "caption": "Там потом ещё много интересного узнаете в процессе👌🏼Главное начать #какпохудеть #какнабратьмассу #ппрецепт #какдобратьбелок ",
        "speech": "нет",
    },
    {
        "n": 5,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@rubezh_education/video/7695800494962265376",
        "caption": "Получите план поступления за границу в 2027🎓 📲 Записывайтесь на бесплатную консультацию — ссылка на запись в Телеграм (30к пдп): rubezhstudy #учебазаграницей #образованиевевропе #бесплатнаяучеба #гран",
        "speech": "нет",
    },
    {
        "n": 6,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@anosof.roman/video/7677840338479762706",
        "caption": "Почему не берут на работу после курсов? 😢 Право на страдания. Что вы думаете, имеет ли человек право сменить профессию в 30 лет, или это инфантилизм, легкомысленность и безалаберность? 🤔 Пишите в комм",
        "speech": "да",
    },
    {
        "n": 7,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@apologygirlrl/video/7695336047642692884",
        "caption": '"Блиин насчет чего мне теперь ныть😭😭😭😭неет нет нас тоже угнетают!!! нам девушки не дают!!! мужчины угнетаются!!!" bruh. men loneliness epidemic не существует, борьба за мужские права бесссысленна, фем',
        "speech": "нет",
    },
    {
        "n": 8,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@niolypolina/video/7690957432595467527",
        "caption": "ТГК Nioly, ты как? В итоге я купила подарок Арине, а как вам туфли? #миллионер #блогер #подарок #агент #аринасавастру ",
        "speech": "да",
    },
    {
        "n": 9,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@niolypolina/video/7692445554780966162",
        "caption": "ТГК Nioly, ты как? Все это обязательно надо сделать перед кастингом в модельное агентство, пользуйтесь✨ #моделинг #модель #агент #селебрити #блогер ",
        "speech": "да",
    },
    {
        "n": 10,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@bebrosechka/video/7694257728126078215",
        "caption": "Знак от Вселенной представляет собой любое случайное событие, фразу, цифру или совпадение, которому человек приписывает особый смысл и воспринимает как подсказку судьбы. В эзотерике и духовных практик",
        "speech": "нет",
    },
    {
        "n": 11,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@culinary_boss/video/7693069044907314453",
        "caption": "Пряный американо со сливочной шапкой 😍🔥 Варю эспрессо в любимой кофеварке Demiand! Арт. WB (белая): WW286274 Арт. WB (черная): WW609011 Арт. OZON: 3392009371 Арт. Яндекс М: 5008360491 Эспрессо - 50 мл",
        "speech": "нет",
    },
    {
        "n": 12,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@alexaprostolegenda/video/7694999379115101458",
        "caption": "Это большая боль. «Ничего не поменяется» «какой смысл» «борьба в пустую» да блять как вы не понимаете, что женщин тысячелетиями угнетали, и все равно не сломили. Если мы смогли добиться того, что есть",
        "speech": "нет",
    },
    {
        "n": 13,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@yngouh/video/7687659564610620679",
        "caption": "Это 5 часть моей истории. Первые две про операцию, 1200 и 2000 мл. третья и четвертая про платных врачей, Минздрав и «сидите на катетеризации». эта - про то, как я перестаю ходить. Мне 18, я хожу с тр",
        "speech": "да",
    },
    {
        "n": 14,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@alexaprostolegenda/video/7694817920488099079",
        "caption": "Не забывайте о том, что если бы месячные были у мужчин, в общественных туалетах были бы бесплатные прокладки ",
        "speech": "нет",
    },
    {
        "n": 15,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@kseniia.fit/video/7694205879561882902",
        "caption": "Тренировка новичку в зале или дома . Выполняй каждое упражнение 3 подхода 10-12 раз #суперзав #техникаупражнений #тренировка #тренировкановичку #новичкам ",
        "speech": "да",
    },
    {
        "n": 16,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@withoutndsss/video/7693535996193049877",
        "caption": "История и персонажи выдуманы. Все совпадения с реальными людьми или ситуациями случайны. Это плод моих фантазий ",
        "speech": "да",
    },
    {
        "n": 17,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@arbyzik803/video/7694414883911240974",
        "caption": "Zhanulka — ток, Жанулька — украинская певица и автор песен, настоящее имя — Жанна Шаталова. Она родилась в 2004 году в Донецке, позже переехала в Киев, а затем в Испанию. В подростковом возрасте начал",
        "speech": "нет",
    },
    {
        "n": 18,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@larax33_/video/7694613556007898388",
        "caption": "У меня этот бас бас бас донбас уже 4 раз в рекомендациях попадается, у меня деградацмя начинается #няшкакотикумри #яненяшкакотик ",
        "speech": "нет",
    },
    {
        "n": 19,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@kseniia.fit/video/7693518441445068054",
        "caption": "Распаковка для ленивых 😁 кто так же ? #суперзав #техникаупражнений #распаковка #распаковкаобуви #кардио ",
        "speech": "да",
    },
    {
        "n": 20,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@laurochka_fit/video/7694823969664355602",
        "caption": "Две тренировки в неделю - это мало? А четыре, из которых ты постоянно пропускаешь две, это много? 😁 Мы часто пытаемся подстроить свою жизнь под идеальную программу. Хотя работать должно наоборот. Рабо",
        "speech": "да",
    },
    {
        "n": 21,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@mahahatrr/video/7694705815604743432",
        "caption": "очень хочу в группу убийцы на барабаны отмечайте @Егор Древлянин  #убийцы #саратов #древлянин #барабанщица #маха",
        "speech": "нет",
    },
    {
        "n": 22,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@7x7_horizontal_russia/video/7694685667107900694",
        "caption": "В Красноярске медики двух роддомов протестуют против их закрытия. Они собирают подписи, записывают обращения к Путину и пытаются добиться отмены решений местного Минздрава. Журналистка «7х7» Светлана ",
        "speech": "да",
    },
    {
        "n": 23,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@missyoureyes17/video/7686867732356238624",
        "caption": "Скучать — это иногда открыть переписку просто посмотреть на имя, переслушать голосовое, зайти на страницу или вспомнить какой-то момент, хотя прекрасно понимаешь, что от этого станет только тяжелее. Э",
        "speech": "нет",
    },
    {
        "n": 24,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@tanyabranding/video/7694341494894775585",
        "caption": "Для мужчин голос женщины - «белый шум».  Общество веками тренировало восприятие: низкий, монотонный баритон считывается как «спокойствие, авторитет и уверенность», а более высокий, динамичный тембр ка",
        "speech": "да",
    },
    {
        "n": 25,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@lizangelwingz/video/7693907470137281824",
        "caption": "Тг ЛИЗА БАТОРИ🩵ответила на долгожданный вопрос приятного аппетита #житуха #еда #рацион #пп #дружба ",
        "speech": "да",
    },
    {
        "n": 26,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@tensor_pn/video/7694643933934095649",
        "caption": "🚨ВПН В ПРОФИЛЕ В России действует Федеральный закон № 281-ФЗ от 31.07.2025. Он накладывает административная ответственность за умышленный поиск и просмотр экстремистских материалов в интернете. #впн #",
        "speech": "да",
    },
    {
        "n": 27,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@bebrosechka/video/7694615756268473608",
        "caption": "Фраза «думай как мужчина» обычно означает предложение использовать стереотипно мужской, рациональный и прямолинейный подход к решению проблем, полностью отключив лишние эмоции. В повседневном общении ",
        "speech": "нет",
    },
    {
        "n": 28,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@sk1ns09/video/7693491403837558034",
        "caption": "Вы Джейн Доу, я Джейн Доу, а вот имена всех 57 человек (на английском, к сожалению) Andrew Hage, Henry Malarkey,Evan Beyers, Johnny Lewis, Matt Luebke, Andrew Solnik, Bennet Mccomb, Danny Rusk, Luke V",
        "speech": "нет",
    },
    {
        "n": 29,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@khurievsk1/video/7694204014317014305",
        "caption": "#fyp #fy #трейдинг  Тут другие правила 1. вам ничего не продают. ни чему не обучают. Только непрошенные советы. 2. у меня открыта ссылка на основу в тгк — написать и спросить что угодно может любой, в",
        "speech": "нет",
    },
    {
        "n": 30,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@depersonalizacia11/video/7693586141785705749",
        "caption": "рецепт: крошка картошка ИНГРЕДИЕНТЫ: картофель  батат (по желанию) петрушка колбаса сыр маринованные огурцы греческий йогурт майонез #рецепт #еда #макулатура #нейтдиаз #крошкакартошка ",
        "speech": "нет",
    },
    {
        "n": 31,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@xeni.fit/video/7694335425648790807",
        "caption": "Ладно, конечно, количество повторений тоже имеет значение 😅 Но магической цифры, после которой мышцы вдруг начинают расти, нет. Можно эффективно тренироваться и на 10, и на 15, и на 20 повторений, есл",
        "speech": "нет",
    },
    {
        "n": 32,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@calistoime/video/7694017064805158152",
        "caption": "Только на первую пару я не хочу на лекцию по комп тех , так как ее я лучше разбираю сама . На ужин мы кушали курицу с макаронами #физмат#влог#мфти#фркт#жизнь",
        "speech": "да",
    },
    {
        "n": 33,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@tensor_pn/video/7693511639026371873",
        "caption": "ВПН В ПРОФИЛЕ. Tensor VPN использует технологию multihop. Благодаря этому РКН понадобиться специально вычислять ip всех наших серверов, что делается только для ведущих крупных сервисов, тогда как серв",
        "speech": "да",
    },
    {
        "n": 34,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@zhvachkam/video/7693994604885970190",
        "caption": "В неполной семье мать смотрит на дочь не как на «маленького принцессу», а как на конкурента, эмоциональное ведро или ресурсодателя. Но СЫНА одинокая женщина подсознательно ставит сына на место «главно",
        "speech": "нет",
    },
    {
        "n": 35,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@calistoime/video/7693618807352364306",
        "caption": "Ну мне уже пофиг, я просто стараюсь говорить себе , что сделала все что смогла #физмат#балл#экзамен#контрольная#подготовка",
        "speech": "нет",
    },
    {
        "n": 36,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@tyloou_/video/7692747013678451975",
        "caption": "Съедобная стиральная капсула 🫧 Биоразлогаемый съедобный пластик: Вода 1 - 250 г Вода 2 - 50 г ( для желатина ) Желатин - 15 г Глицерин - 7 г Глюкозный сироп - 12 г Агар - 1  Каппа - 3 г 1. Замочи жела",
        "speech": "да",
    },
    {
        "n": 37,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@shima_shima25/video/7693209679651933448",
        "caption": "I AM JANE DOE  Джейн Доу (Jane Doe) — это стандартное английское условное обозначение для женщины, чья личность неизвестна или которая по какой-то причине хочет остаться анонимной. Вот основные случаи",
        "speech": "да",
    },
    {
        "n": 38,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@silence.moid/video/7692821484670094644",
        "caption": "А ещё у меня есть недвижимость и высшее образование #высшееобразование #недвижимость #беливиксволшебныйдар ",
        "speech": "нет",
    },
    {
        "n": 39,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@jopa297478392/video/7692474654287613191",
        "caption": "как бы звучала макулатура, если бы была хорошей группой #макулатура #публикавыбираетсама #эхопрокуренныхподъездов #эпп",
        "speech": "нет",
    },
    {
        "n": 40,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@elinprofilm/video/7693066488541924640",
        "caption": "9 октября выйдет жуткий хоррор от создателей «Обсессия»🍿 как вам сюжет? #обсессия #другаямама #триллер #хоррор #чтопосмотреть ",
        "speech": "да",
    },
    {
        "n": 41,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@kischuk_trainer/video/7693175100778007826",
        "caption": "Не забудь сохранить мою тренировку и попробовать ее🫶🏼Запись ко мне на тренировки/ведение—Директ🫶🏼",
        "speech": "нет",
    },
    {
        "n": 42,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@ksirkii/video/7692745520191687943",
        "caption": "любимая пара в интернете❤️❤️❤️ #стафанов #штефанов #александрштефанов #стасайкакпросто  #триметранадуровнемнеба #fyp ",
        "speech": "нет",
    },
    {
        "n": 43,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@bishhhhk/video/7692833246891101448",
        "caption": "Митинги снова появляются в политическом постранстве. Сегодня разбираюсь в новом симтоме дисфункциональности управления.  #россия #митинг #протесты #выборы",
        "speech": "да",
    },
    {
        "n": 44,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@love.tasty.food/video/7691810599256263956",
        "caption": "Пока кто-то жалуется на работу, я жалуюсь на её отсутствие 😭 И хочу поскорее найти свою команду!  #it #фронтендразработчик #поискработы #вакансии #вакансииit",
        "speech": "да",
    },
    {
        "n": 45,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@logvinarkad_/video/7691325446121753876",
        "caption": "«УЛИЦА ГОВОРИТ», а ты что скажешь?  Видео создано при поддержки @Криптоплатформа Dzengi  #аркадьлогвин #улицаговорит #тренд #интервью #отношения ",
        "speech": "да",
    },
    {
        "n": 46,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@fedor.mencha/video/7691628151364996372",
        "caption": "Записаться на онлайн ведение можно по ссылке в моём профиле, либо через личные сообщения🔝✅ #похудение #тренировка #фитнес #здоровье #спорт ",
        "speech": "да",
    },
    {
        "n": 47,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@electrojoint/video/7691682582693514504",
        "caption": "помогите зафорсить трек! вся инфа о нем в тгк: elctrojoint #recommendations #рекомендации #врекомендации #глобальныерекомендации #fyp",
        "speech": "нет",
    },
    {
        "n": 48,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@mother.yukionna/video/7691034568341687584",
        "caption": "Женщин ждет прекрасный мир, а моидов контроль и подчинение🤩#матриархат #радфем #радикальныйфеминизм #женщины #щп ",
        "speech": "нет",
    },
    {
        "n": 49,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@myr_16/video/7690493287655869717",
        "caption": "Хорошо, что мне 44. Уже не нужно никому ничего доказывать. Только себе — что могу ещё больше. 💪🔥#казахстан🇰🇿 #экибастуз #фитнес #любисебя #мотивация Только себе — что могу ещё больше. 💪🔥",
        "speech": "нет",
    },
    {
        "n": 50,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@fedor.mencha/video/7691036014235618612",
        "caption": "Записаться на онлайн ведение можно по ссылке в моём профиле, либо через личные сообщения🔝✅ #похудение #фитнес #тренировка #зож #спорт ",
        "speech": "да",
    },
    {
        "n": 51,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@boss_neuro/video/7688352064287739156",
        "caption": "Мать была уверена, что её секрет никто никогда не узнает. Но один школьный тест поставил под сомнение всё. Теперь отец собирается докопаться до правды — и она понимает, что времени почти не осталось.",
        "speech": "да",
    },
    {
        "n": 52,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@vilokaa/video/7691286520149445906",
        "caption": "Сейчас мой отец - прекрасный и хороший человек. Я давно простила его за моё детство. Он не пьёт, ведёт здоровый образ жизни, часто звонит и заботится обо мне. Я благодарна ему за это.",
        "speech": "нет",
    },
    {
        "n": 53,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@neurozeh.ru/video/7689452343452110101",
        "caption": "ИИ может написать грамотно, логично и по всем правилам, но именно эта правильность иногда делает текст безжизненным. Проблема не в том, чтобы заставить нейросеть писать «как человек», а в том, чтобы н",
        "speech": "да",
    },
    {
        "n": 54,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@standuplokal/video/7689091198753033494",
        "caption": "Вышел Пораразби из Варшавы! Смотрите выпуск целиком на нашемканале lokal #стендапкомики #стендапшоу #стендап #подкаст",
        "speech": "да",
    },
    {
        "n": 55,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@her_idea_369/video/7690997366161427732",
        "caption": "Вам не нужно разделять мою позицию, можете думать как угодно, но приходить ко мне ныть, что я должна изменить свою позицию, пускать сопли и требовать от меня выйти с вами на дискуссию НЕТ. Я давно при",
        "speech": "нет",
    },
    {
        "n": 56,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@faviee31/video/7690642149519396116",
        "caption": "ладно, вы классные. +ЛЕЕ И РЕЗИНОВЫЙ, успокойтесь  #творожныйбизнес #руреп #рек #recommendations #on @тг: творожный бизнес творожный бизнес — это не про молочное производство, а музыкальный проект. За",
        "speech": "нет",
    },
    {
        "n": 57,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@pink__angel_gotovit/video/7072020781319851265",
        "caption": "Подробный рецепт в inst: pink__angel_        #сырники #сырникибезсахара #сырникибезяиц #сырникиза5минут #сырникирецепт #сырникиназавтрак #сырнички #сырникибезмуки",
        "speech": "да",
    },
    {
        "n": 58,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@liza_belis/video/7690837170378018080",
        "caption": "не смешно, нелепо, глупо, вторично, безосновательно, неразумно, бессмысленно, непонятно, отнюдь не остроумно ##лиза_белис #рофл #постирония #щп #67 ",
        "speech": "да",
    },
    {
        "n": 59,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@ssilensio_blog/video/7690879905155927310",
        "caption": "Скрещиваю картошку и роллы, чтобы подтвердить, что любое блюдо может быть сделано из картошки! Так что эти роллы на 90% состоят из картошки  #ssilensio #еда #рецепт #роллы #картошка ",
        "speech": "да",
    },
    {
        "n": 60,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@jack_napes/video/7669759178264612116",
        "caption": "В Токмачке классно Если залетит сделаю фулл Ма́лая Токма́чка (укр. Мала́ Токма́чка) — село в Пологовском районе Запорожской области Украины, административный центр Малотокмачанской сельской общины. До",
        "speech": "нет",
    },
    {
        "n": 61,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@gonzo_authorka/video/7690120228876995848",
        "caption": "В каком году родился самый несчастный зумер? Если обобщать вводы двух видео, то можно сказать, что чем раньше родился — тем лучше. Все это очень печально. А вы согласны? ",
        "speech": "да",
    },
    {
        "n": 62,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@ldrphrt/video/7690956640887115029",
        "caption": "6 лет подряд каждую осень один и тот же срач, я обожаю. Видео Миланы Некрасовой у меня не вызывает негатива, я думаю она просто рейджбайтнула и люди начали выдавать хиханьки хаханьки. Вопросы только к",
        "speech": "нет",
    },
    {
        "n": 63,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@maslova.trainer/video/7689009273715576072",
        "caption": "В какой-то момент понимаешь, что лучший движ — тот, после которого тебе хорошо и сегодня, и завтра утром! И нет — это не про скучную жизнь. Просто теперь кайф выглядит немного иначе: тренировка → вкус",
        "speech": "нет",
    },
    {
        "n": 64,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@7345577959379731462/video/7685057872371141908",
        "caption": "Моноколесо — это персональное электрическое транспортное средство с одним колесом, двумя боковыми платформами для ног и системой автоматического удержания равновесия. Основные части: * Колесо и шина —",
        "speech": "нет",
    },
    {
        "n": 65,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@addanilovaa/video/7695184332280876294",
        "caption": "Моя тренировка на ягодицы и зпб ⚒️   // на мне майка vivclub.ru 🤍   • в румынской тяге делаю 2 разминочных подхода - с 50% рабочего веса, и с 75% • в ягодичном мостике - 1 разминочный подход с 50% раб",
        "speech": "да",
    },
    {
        "n": 66,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@annturka/video/7665324929747029255",
        "caption": "рецепт пп-синнабонов🤌🏻💔 Ингредиенты: Тесто: творог 300 г 3 яйца разрыхлитель 10 г Мука цельнозерновая ≈200 гр (ориентируюсь на консистенцию теста) Сахзам 2 мерных ложки Щепотка соли Начинка: корица 4-",
        "speech": "да",
    },
    {
        "n": 67,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@etolerika/video/7694959401794424094",
        "caption": "Подборка еды, которая будет полезна для красоты кожи, волос и ногтей. Част 2 💌 @все это лерика  #заботаосебе #здоровоетело #уходзакожей #полезныесоветы #полезнаяеда ",
        "speech": "да",
    },
    {
        "n": 68,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@maria_homechef/video/7693556857654250773",
        "caption": "Творожные ватрушки 🍩  - 300г творога (у меня 1%) - 1 яйцо  - 90 грамм рисовой муки (или 80 пшеничной)  - ванилин  - 1 ст.л сахара  - щепотка соли  - 1/2 ч.л. разрыхлителя  начинка:  - 200 грамм мягког",
        "speech": "нет",
    },
    {
        "n": 69,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@sitora.20/video/7695621933588679943",
        "caption": "Картофельная запеканка с курицей и грибами Ингредиенты: * Картофель — 3 шт. (1 большая и 2 средние) * Молоко — 2 ст. л. * Куриное филе — 1 шт. * Лук — ½ шт. * Шампиньоны — 3–4 шт. * Сливки 20% — 50 мл",
        "speech": "да",
    },
    {
        "n": 70,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@lraresnake/video/7695875950617152788",
        "caption": "наконец-то монополисты получили по заслугам #путинхуйло #fyp #нетвойне #мирукраине #свободуроссии",
        "speech": "нет",
    },
    {
        "n": 71,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@superzav_fit/video/7691005131298491651",
        "caption": "Факты в отношениях 12 лет . Что для вас стрем  ? #отношенияспарнем #любовьотношения #мерзав #шокфакты ",
        "speech": "да",
    },
    {
        "n": 72,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@tanyabranding/video/7694731464369343777",
        "caption": "Женщин заставили больше страдать и рисковать жизнью, чтобы мужчинам было удобнее смотреть ",
        "speech": "да",
    },
    {
        "n": 73,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@kischuk_trainer/video/7693518226281516295",
        "caption": "Запись ко мне на тренировки-онлайн/ведение—директ💌 Еще больше тренировок/жизни у меня в тгк kischuk ",
        "speech": "нет",
    },
    {
        "n": 74,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@comigration/video/7690952694558772501",
        "caption": "Эксклюзивный Вотафлаг из Алматы №2 на наших платных платформах! #шашлык #девушки #отношения",
        "speech": "да",
    },
    {
        "n": 75,
        "platform": "TikTok",
        "url": "https://www.tiktok.com/@fanstef7/video/7696245465880202518",
        "caption": "Я ее знаю, завидуйте @Аля Шнуркова #штефанов #котик #партияпчеловодов #александрштефанов ",
        "speech": "да",
    },
    {
        "n": 76,
        "platform": "VK",
        "url": "https://vk.com/clip-194778748_456240278",
        "caption": "Clip by @onlinebankrot",
        "speech": "да",
    },
    {
        "n": 77,
        "platform": "VK",
        "url": "https://vk.com/clip-212781631_456239710",
        "caption": "Clip by @veronika_kromberger",
        "speech": "да",
    },
    {
        "n": 78,
        "platform": "VK",
        "url": "https://vk.com/clip-219478341_456239272",
        "caption": "Clip by @artworx_crew",
        "speech": "да",
    },
    {
        "n": 79,
        "platform": "VK",
        "url": "https://vk.com/clip-211006595_456239084",
        "caption": "Clip by @mashaa_zorinaa",
        "speech": "да",
    },
    {
        "n": 80,
        "platform": "VK",
        "url": "https://vk.com/clip-179784060_456244300",
        "caption": "Clip by @moskva7777",
        "speech": "да",
    },
    {
        "n": 81,
        "platform": "VK",
        "url": "https://vk.com/clip-231414357_456239285",
        "caption": "Clip by @pravodengi",
        "speech": "да",
    },
    {
        "n": 82,
        "platform": "VK",
        "url": "https://vk.com/clip-99173315_456282046",
        "caption": "Clip by @don24tv",
        "speech": "да",
    },
    {
        "n": 83,
        "platform": "VK",
        "url": "https://vk.com/clip-228626101_456239235",
        "caption": "Clip by @s_detskogo_na_vzrosliy",
        "speech": "да",
    },
    {
        "n": 84,
        "platform": "VK",
        "url": "https://vk.com/clip-219199731_456239320",
        "caption": "Clip by @dorgear",
        "speech": "да",
    },
    {
        "n": 85,
        "platform": "VK",
        "url": "https://vk.com/clip-185334713_456249908",
        "caption": "Clip by @ant1publ1c",
        "speech": "да",
    },
    {
        "n": 86,
        "platform": "VK",
        "url": "https://vk.com/clip-98368576_456252067",
        "caption": "Clip by @maksimenko_photoschool",
        "speech": "да",
    },
    {
        "n": 87,
        "platform": "VK",
        "url": "https://vk.com/clip-160940586_456239666",
        "caption": "Clip by @dymdymov",
        "speech": "да",
    },
    {
        "n": 88,
        "platform": "VK",
        "url": "https://vk.com/clip-211423460_456239747",
        "caption": "Clip by @antonvolkov",
        "speech": "да",
    },
    {
        "n": 89,
        "platform": "VK",
        "url": "https://vk.com/clip-191498811_456241633",
        "caption": "Clip by @doctor_ulik",
        "speech": "да",
    },
    {
        "n": 90,
        "platform": "VK",
        "url": "https://vk.com/clip-212677051_456240481",
        "caption": "Clip by @afterschoolway",
        "speech": "да",
    },
    {
        "n": 91,
        "platform": "VK",
        "url": "https://vk.com/clip-39543596_456239409",
        "caption": "Clip by @clubplyushki",
        "speech": "да",
    },
]

QUERIES = [
    {
        "q": "айтишник рассказывает про эксперимент",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [1],
    },
    {
        "q": "люди умерли из-за бага",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [1],
    },
    {
        "q": "ролик, где айтишник пугает, что из-за багов в коде могут пострадать люди",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [1],
    },
    {
        "q": "медицинский аппарат, который из-за ошибки в программе облучал пациентов",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [1],
    },
    {
        "q": "план поступления в зарубежный вуз",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [5],
    },
    {
        "q": "почему не берут на работу",
        "kind": "тема",
        "where": "речь",
        "author": "Анастасия",
        "answers": [6, 44],
    },
    {
        "q": "рынок труда в айти",
        "kind": "тема",
        "where": "речь",
        "author": "Анастасия",
        "answers": [6, 44],
    },
    {
        "q": "идеальный портрет кандидата",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [6],
    },
    {
        "q": "идеальный работник",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [6],
    },
    {
        "q": "выиграл все олимпиады",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [6],
    },
    {
        "q": "трудоустройство после курсов",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [6],
    },
    {
        "q": "смена профессии",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [6],
    },
    {
        "q": "онлайн-курсы и работа",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [6],
    },
    {
        "q": "почему после курсов не берут на работу в IT",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [6],
    },
    {
        "q": "не поздно ли менять профессию ближе к тридцати",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [6],
    },
    {
        "q": "блогерша выбрала подруге подарок, туфли",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [8],
    },
    {
        "q": "как подготовиться к пробам в модели",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [9],
    },
    {
        "q": "как сварить кофе с пряностями и сливками",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [11],
    },
    {
        "q": "в 18 лет после операций перестала ходить",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [13],
    },
    {
        "q": "тренировка для маленьких",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [15],
    },
    {
        "q": "тренировка на всё тело",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [15],
    },
    {
        "q": "тренировка для новичка",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [15],
    },
    {
        "q": "какой вес подобрать",
        "kind": "тема",
        "where": "речь",
        "author": "Анастасия",
        "answers": [15, 65],
    },
    {
        "q": "как делать присед",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [15],
    },
    {
        "q": "техника приседа",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [15],
    },
    {
        "q": "простые упражнения для тех, кто только начинает заниматься",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [15],
    },
    {
        "q": "девушка распаковывает посылку со спортивной одеждой",
        "kind": "перефраз",
        "where": "изображение",
        "author": "Claude (черновик)",
        "answers": [19],
    },
    {
        "q": "сколько надо тренировок в неделю",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [20],
    },
    {
        "q": "достаточное количество тренировок в неделю",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [20],
    },
    {
        "q": "как выстроить тренировочный план",
        "kind": "тема",
        "where": "речь",
        "author": "Анастасия",
        "answers": [20, 65],
    },
    {
        "q": "что делать, если часто пропускаю тренировки",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [20],
    },
    {
        "q": "тренер объясняет, что пары походов в зал в неделю вполне хватает",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [20],
    },
    {
        "q": "лучше заниматься реже, но не пропускать, чем по плану, который не выполняешь",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [20],
    },
    {
        "q": "протест медиков",
        "kind": "тема",
        "where": "речь",
        "author": "Анастасия",
        "answers": [22, 89],
    },
    {
        "q": "закрытие роддома",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [22],
    },
    {
        "q": "несправедливость властей",
        "kind": "тема",
        "where": "речь",
        "author": "Анастасия",
        "answers": [22, 43],
    },
    {
        "q": "закрытие роддома в Красноярске",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [22],
    },
    {
        "q": "протест беременных и матерей",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [22],
    },
    {
        "q": "комфортные роды",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [22],
    },
    {
        "q": "врачи протестуют против закрытия роддома",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [22],
    },
    {
        "q": "видео, где медики борются, чтобы их родильный дом не закрыли",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [22],
    },
    {
        "q": "в Красноярске врачи против того, чтобы закрыли больницу для рожениц",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [22],
    },
    {
        "q": "когда скучаешь по человеку и перечитываешь его переписку",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [23],
    },
    {
        "q": "почему мужчины хуже слышат, когда говорит женщина",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [24],
    },
    {
        "q": "рецепт, спетый как песня",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [30],
    },
    {
        "q": "сколько повторений нужно, чтобы росли мышцы",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [31, 15],
    },
    {
        "q": "студенческий влог: прогуляла лекцию, на ужин курица с макаронами",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [32],
    },
    {
        "q": "съедобная капсула для стирки из желатина",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [36],
    },
    {
        "q": "анонс страшного фильма, который скоро выйдет в кино",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [40],
    },
    {
        "q": "девушка не может найти работу и ищет команду",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [44],
    },
    {
        "q": "девушка рисует на улице и берёт интервью у прохожих",
        "kind": "перефраз",
        "where": "изображение",
        "author": "Claude (черновик)",
        "answers": [45],
    },
    {
        "q": "фитнес-тренер приглашает на онлайн-ведение",
        "kind": "тема",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [46, 50, 73, 41],
    },
    {
        "q": "женщина в 44 года тренируется и никому ничего не доказывает",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [49],
    },
    {
        "q": "сериал, сделанный нейросетью, про мать, которая скрывает секрет",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [51],
    },
    {
        "q": "как отличить ИИ-текст от человеческого",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [53],
    },
    {
        "q": "детектор ИИ-текста",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [53],
    },
    {
        "q": "как пишет ИИ",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [53],
    },
    {
        "q": "как понять, что текст написан нейросетью",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [53],
    },
    {
        "q": "как пишет нейросеть",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [53],
    },
    {
        "q": "как разные LLM пишут тексты",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [53],
    },
    {
        "q": "правила написания текста с ИИ",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [53],
    },
    {
        "q": "как сделать текст от нейросети живым",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [53],
    },
    {
        "q": "нонбайнари",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [54],
    },
    {
        "q": "концепция гендеров",
        "kind": "тема",
        "where": "речь",
        "author": "Анастасия",
        "answers": [54, 27],
    },
    {
        "q": "стендап Долгополов",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [54],
    },
    {
        "q": "стендап Пораразби",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [54],
    },
    {
        "q": "стендап про нонбайнари",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [54],
    },
    {
        "q": "стендап про небинарного парня на плакате",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [54],
    },
    {
        "q": "роллы, которые почти целиком из картошки",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [59],
    },
    {
        "q": "блюда из картошки",
        "kind": "тема",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [59, 30, 69],
    },
    {
        "q": "рассуждения о том, какому поколению молодёжи не повезло больше всего",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [61],
    },
    {
        "q": "бабушка катается на моноколесе",
        "kind": "перефраз",
        "where": "изображение",
        "author": "Claude (черновик)",
        "answers": [64],
    },
    {
        "q": "тренировка на ягодицы",
        "kind": "тема",
        "where": "речь",
        "author": "Анастасия",
        "answers": [65, 2],
    },
    {
        "q": "тренировочный план",
        "kind": "тема",
        "where": "речь",
        "author": "Анастасия",
        "answers": [65, 20],
    },
    {
        "q": "тренировка без квадрицепсов",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [65],
    },
    {
        "q": "тренировка ягодиц: румынская тяга и ягодичный мостик",
        "kind": "тема",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [65, 2],
    },
    {
        "q": "полезные булочки с корицей без сахара",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [66],
    },
    {
        "q": "что приготовить из творога",
        "kind": "тема",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [66, 68, 57],
    },
    {
        "q": "продукты, от которых будут красивые волосы и ногти",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [67],
    },
    {
        "q": "выпечка из творога без пшеничной муки",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [68],
    },
    {
        "q": "картошка с мясом и шампиньонами в духовке",
        "kind": "перефраз",
        "where": "текст",
        "author": "Claude (черновик)",
        "answers": [69],
    },
    {
        "q": "пара, которая очень долго вместе, рассказывает про свои отношения",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [71],
    },
    {
        "q": "носила вещи любовницы папы",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "детство в нищете",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "носила чужие вещи",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "нищее детство",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "жизнь в Сибири",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "кормили родители одноклассников",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "девушка рассказывает про нищее детство",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "школа конного спорта в Новосибирске",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "конный спорт",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "бесплатная школа конного спорта в Новосибирске",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "травмы с деньгами",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [77],
    },
    {
        "q": "девушка рассказывает, как в детстве ей приходилось носить чужие вещи",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [77],
    },
    {
        "q": "девушка вспоминает бедное детство в Сибири без зимней обуви",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [77],
    },
    {
        "q": "рыбья чешуя дорогая",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [85],
    },
    {"q": "коллаген", "kind": "перефраз", "where": "речь", "author": "Анастасия", "answers": [85]},
    {
        "q": "ролик про то, что чешуя от рыбы стоит огромных денег",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [85],
    },
    {
        "q": "в Китае зарабатывают на том, что остаётся после разделки рыбы",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [85],
    },
    {
        "q": "больница изменила условия работы",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [89],
    },
    {
        "q": "врачам изменили условия работы, вместо 33 ч теперь 39 ч",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [89],
    },
    {
        "q": "что изменилось в работе у врачей",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [89],
    },
    {
        "q": "11 минут на пациента",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [89],
    },
    {
        "q": "уволили врача из-за повышения рабочей ставки и протеста против этого",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [89],
    },
    {
        "q": "врачу увеличили часы приёма с 33 до 39 без прибавки к зарплате",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [89],
    },
    {
        "q": "звук вертолёта стендап",
        "kind": "перефраз",
        "where": "речь",
        "author": "Анастасия",
        "answers": [91],
    },
    {
        "q": "двое коллег соревнуются, кто смешнее изобразит вертолёт",
        "kind": "перефраз",
        "where": "речь",
        "author": "Claude (черновик)",
        "answers": [91],
    },
    {
        "q": "как поменять масло в машине",
        "kind": "без ответа",
        "where": "—",
        "author": "Claude (черновик)",
        "answers": [],
    },
    {
        "q": "обзор нового смартфона",
        "kind": "без ответа",
        "where": "—",
        "author": "Claude (черновик)",
        "answers": [],
    },
    {
        "q": "рецепт борща",
        "kind": "без ответа",
        "where": "—",
        "author": "Claude (черновик)",
        "answers": [],
    },
]

print(len(CLIPS), "роликов,", len(QUERIES), "запросов")

In [ ]:
# Скачиваем только аудио (уже скачанные пропускаются — ячейку можно перезапускать)
import os
import subprocess

os.makedirs("audio", exist_ok=True)
failed = []
for c in CLIPS:
    n = c["n"]
    if os.path.exists(f"audio/{n:02d}.mp3"):
        continue
    r = subprocess.run(
        [
            "yt-dlp",
            "-q",
            "-f",
            "bestaudio/best",
            "-x",
            "--audio-format",
            "mp3",
            "-o",
            f"audio/{n:02d}.%(ext)s",
            c["url"],
        ],
        capture_output=True,
        text=True,
    )
    if not os.path.exists(f"audio/{n:02d}.mp3"):
        failed.append(n)
        print(f"{n:>3} ОШИБКА: {(r.stderr.strip().splitlines() or ['?'])[-1][:120]}")
print("скачано", len(CLIPS) - len(failed), "из", len(CLIPS), "; без аудио:", failed)

In [ ]:
# Распознавание речи: faster-whisper large-v3
# Аудио читаем через ffmpeg: свежий PyAV в Colab несовместим с faster-whisper.
# Фильтр тишины (VAD) на речи поверх музыки вырезает голос (MEM-16) — порог ослаблен, при пустом результате распознаём без фильтра.
import json
import time

import numpy as np
import torch
from faster_whisper import WhisperModel


def load_audio(path, sr=16000):
    out = subprocess.run(
        ["ffmpeg", "-nostdin", "-i", path, "-f", "s16le", "-ac", "1", "-ar", str(sr), "-"],
        capture_output=True,
        check=True,
    ).stdout
    return np.frombuffer(out, np.int16).astype(np.float32) / 32768.0


asr = WhisperModel(
    "large-v3", device=device, compute_type="float16" if device == "cuda" else "int8"
)
os.makedirs("transcripts", exist_ok=True)
TR = {}
t_all = time.time()
for c in CLIPS:
    n = c["n"]
    path = f"transcripts/{n:02d}.json"
    if os.path.exists(path):
        TR[n] = json.load(open(path, encoding="utf-8"))
        continue
    if not os.path.exists(f"audio/{n:02d}.mp3"):
        continue
    audio = load_audio(f"audio/{n:02d}.mp3")
    segs, info = asr.transcribe(
        audio, language="ru", beam_size=5, vad_filter=True, vad_parameters=dict(threshold=0.3)
    )
    segs = list(segs)
    if not segs:
        segs, info = asr.transcribe(audio, language="ru", beam_size=5, vad_filter=False)
        segs = list(segs)
    TR[n] = dict(
        clip_id=n,
        model="faster-whisper large-v3",
        language=info.language,
        duration=round(info.duration, 2),
        segments=[
            dict(start=round(s.start, 2), end=round(s.end, 2), text=s.text.strip()) for s in segs
        ],
    )
    json.dump(TR[n], open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"{n:>3} {info.duration:5.0f} с | {' '.join(s.text.strip() for s in segs)[:100]}")
print(f"распознано {len(TR)} роликов за {time.time() - t_all:.0f} с")
# освобождаем память под модели эмбеддингов и реранкер (иначе Colab падает от нехватки памяти)
import gc  # noqa: E402

del asr
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Разбиение на фрагменты: окно по времени с перекрытием; подпись ролика — отдельный фрагмент
import re


def clean_caption(s):
    s = re.sub(r"#\S+", " ", s or "")
    s = re.sub(r"^Clip by @\S+$", "", s.strip())
    return re.sub(r"\s+", " ", s).strip()


def chunks_for(n, window, stride):
    out = []
    cap = clean_caption(next(c["caption"] for c in CLIPS if c["n"] == n))
    if cap:
        out.append(dict(clip=n, kind="подпись", start=None, end=None, text=cap[:1000]))
    segs = TR.get(n, {}).get("segments", [])
    if not segs:
        return out
    if window is None:  # весь транскрипт одним фрагментом
        return out + [
            dict(
                clip=n,
                kind="речь",
                start=segs[0]["start"],
                end=segs[-1]["end"],
                text=" ".join(s["text"] for s in segs),
            )
        ]
    t, last = 0.0, segs[-1]["end"]
    while True:
        part = [s for s in segs if s["end"] > t and s["start"] < t + window]
        if part and (
            not out or out[-1]["text"] != " ".join(s["text"] for s in part)
        ):  # длинные сегменты дают одинаковые окна
            out.append(
                dict(
                    clip=n,
                    kind="речь",
                    start=part[0]["start"],
                    end=part[-1]["end"],
                    text=" ".join(s["text"] for s in part),
                )
            )
        if t + window >= last:
            break
        t += stride
    return out


# варианты разбиения: (название, окно, шаг), секунды
CHUNKINGS = [
    ("окно 15 с / шаг 7 с", 15, 7),
    ("окно 30 с / шаг 15 с", 30, 15),
    ("окно 60 с / шаг 30 с", 60, 30),
    ("весь транскрипт", None, None),
]
CHUNKS = {name: [ch for c in CLIPS for ch in chunks_for(c["n"], w, s)] for name, w, s in CHUNKINGS}
for name, ch in CHUNKS.items():
    print(
        f"{name:22} {len(ch):4} фрагментов, в среднем {np.mean([len(x['text'].split()) for x in ch]):.0f} слов"
    )

In [ ]:
# Метрики: ролик считается найденным, если любой из правильных роликов попал в первые k
ALL = [c["n"] for c in CLIPS]
EVAL = [q for q in QUERIES if q["answers"]]
NOANS = [q for q in QUERIES if not q["answers"]]


def rank_clips(chunk_scores, chunks):
    best = {}
    for s, ch in zip(chunk_scores, chunks, strict=True):
        if s > best.get(ch["clip"], -1e9):
            best[ch["clip"]] = float(s)
    order = sorted(ALL, key=lambda n: -best.get(n, -1e9))
    return order, best


def metrics(ranks):
    r = np.array([x if x else 10**6 for x in ranks])
    return {
        "Recall@1": np.mean(r <= 1),
        "Recall@5": np.mean(r <= 5),
        "Recall@10": np.mean(r <= 10),
        "MRR": np.mean(1 / r),
    }


def first_rank(order, answers):
    return next((i + 1 for i, n in enumerate(order) if n in answers), None)

In [ ]:
# Модели эмбеддингов. У e5 обязательны префиксы «query: » и «passage: », у BGE-M3 префиксов нет.
import gc

from sentence_transformers import SentenceTransformer

MODELS = {
    "multilingual-e5-large": dict(
        name="intfloat/multilingual-e5-large", q="query: ", p="passage: "
    ),
    "bge-m3": dict(name="BAAI/bge-m3", q="", p=""),
}
rows, per_query, SIMS = [], [], {}
for mname, cfg in MODELS.items():
    t0 = time.time()
    model = SentenceTransformer(cfg["name"], device=device)
    load = time.time() - t0
    if device == "cuda":
        model.half()
    dim = model.get_sentence_embedding_dimension()
    t0 = time.time()
    qv = model.encode(
        [cfg["q"] + q["q"] for q in QUERIES], normalize_embeddings=True, batch_size=64
    )
    q_ms = (time.time() - t0) / len(QUERIES) * 1000
    for cname, chunks in CHUNKS.items():
        t0 = time.time()
        pv = model.encode(
            [cfg["p"] + ch["text"] for ch in chunks], normalize_embeddings=True, batch_size=32
        )
        enc = time.time() - t0
        sims = qv @ pv.T
        SIMS[(mname, cname)] = sims
        ranks, top_noans, top_ans = [], [], []
        for qi, q in enumerate(QUERIES):
            order, best = rank_clips(sims[qi], chunks)
            if q["answers"]:
                rk = first_rank(order, q["answers"])
                ranks.append(rk)
                top_ans.append(best[order[0]])
                per_query.append(
                    dict(
                        model=mname,
                        chunking=cname,
                        query=q["q"],
                        author=q["author"],
                        where=q["where"],
                        answers=q["answers"],
                        rank=rk,
                        top3=order[:3],
                    )
                )
            else:
                top_noans.append(best[order[0]])
        m = metrics(ranks)
        rows.append(
            dict(
                модель=mname,
                разбиение=cname,
                **{k: round(v, 3) for k, v in m.items()},
                **{
                    "размер вектора": dim,
                    "фрагментов": len(chunks),
                    "векторы фрагментов, с": round(enc, 1),
                    "фрагментов в с": round(len(chunks) / enc),
                    "запрос, мс": round(q_ms, 1),
                    "загрузка модели, с": round(load),
                    "ср. сходство топ-1 (есть ответ)": round(float(np.mean(top_ans)), 3),
                    "ср. сходство топ-1 (нет ответа)": round(float(np.mean(top_noans)), 3),
                },
            )
        )
        print(
            f"{mname:22} {cname:22} Recall@10={m['Recall@10']:.3f} MRR={m['MRR']:.3f}  ({enc:.1f} с на {len(chunks)} фрагм.)"
        )
    del model, pv
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Для сравнения: поиск по словам (BM25) — то, что даст полнотекстовый поиск без моделей
from rank_bm25 import BM25Okapi


def tok(s):
    return re.findall(r"\w+", s.lower())


for cname, chunks in CHUNKS.items():
    bm = BM25Okapi([tok(ch["text"]) for ch in chunks])
    ranks = [
        first_rank(rank_clips(bm.get_scores(tok(q["q"])), chunks)[0], q["answers"]) for q in EVAL
    ]
    m = metrics(ranks)
    rows.append(
        dict(модель="BM25 (по словам)", разбиение=cname, **{k: round(v, 3) for k, v in m.items()})
    )
    print(f"{'BM25':22} {cname:22} Recall@10={m['Recall@10']:.3f} MRR={m['MRR']:.3f}")

In [ ]:
# Реранкер: берём 50 лучших фрагментов векторного поиска и пересортировываем их моделью bge-reranker-v2-m3,
# которая читает запрос и фрагмент вместе. Ролики вне этих 50 идут следом в порядке векторного поиска.
from sentence_transformers import CrossEncoder

K = 50
rr = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=device)
if device == "cuda":
    rr.model.half()
for (mname, cname), sims in SIMS.items():
    chunks = CHUNKS[cname]
    tops = {qi: np.argsort(-sims[qi])[:K] for qi, q in enumerate(QUERIES) if q["answers"]}
    pairs = [(QUERIES[qi]["q"], chunks[i]["text"]) for qi, top in tops.items() for i in top]
    t0 = time.time()
    scores = rr.predict(pairs, batch_size=16, show_progress_bar=False)
    dt = time.time() - t0
    ranks, pos = [], 0
    for qi, top in tops.items():
        sc = scores[pos : pos + len(top)]
        pos += len(top)
        best = {}
        for s, i in zip(sc, top, strict=True):
            n = chunks[i]["clip"]
            if s > best.get(n, -1e9):
                best[n] = float(s)
        vec_order, _ = rank_clips(sims[qi], chunks)
        order = sorted(best, key=lambda n: -best[n]) + [n for n in vec_order if n not in best]
        q = QUERIES[qi]
        rk = first_rank(order, q["answers"])
        ranks.append(rk)
        per_query.append(
            dict(
                model=mname + " + реранкер",
                chunking=cname,
                query=q["q"],
                author=q["author"],
                where=q["where"],
                answers=q["answers"],
                rank=rk,
                top3=order[:3],
            )
        )
    m = metrics(ranks)
    rows.append(
        dict(
            модель=mname + " + реранкер",
            разбиение=cname,
            **{k: round(v, 3) for k, v in m.items()},
            **{"реранкер на запрос (GPU), мс": round(dt / len(tops) * 1000)},
        )
    )
    print(
        f"{mname + ' + реранкер':34} {cname:22} Recall@10={m['Recall@10']:.3f} MRR={m['MRR']:.3f}  ({dt / len(tops) * 1000:.0f} мс на запрос)"
    )
del rr
gc.collect()
torch.cuda.empty_cache()

# Сервер, скорее всего, без GPU: меряем реранкер на CPU на 5 запросах (требование — ответ за 2 с)
rr_cpu = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cpu")
chunks = CHUNKS["окно 30 с / шаг 15 с"]
sims = SIMS[("bge-m3", "окно 30 с / шаг 15 с")]
t0 = time.time()
for qi in range(5):
    rr_cpu.predict(
        [(QUERIES[qi]["q"], chunks[i]["text"]) for i in np.argsort(-sims[qi])[:K]],
        batch_size=16,
        show_progress_bar=False,
    )
RERANK_CPU_MS = round((time.time() - t0) / 5 * 1000)
print(f"реранкер на CPU: {RERANK_CPU_MS} мс на запрос ({K} фрагментов)")
rows.append(
    dict(
        модель="реранкер на CPU (замер)",
        разбиение="окно 30 с / шаг 15 с",
        **{"реранкер на запрос (CPU), мс": RERANK_CPU_MS},
    )
)
del rr_cpu
gc.collect()

In [ ]:
# Итоговая таблица и разрезы
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
res = pd.DataFrame(rows)
display(res.sort_values(["Recall@10", "MRR"], ascending=False))

pq = pd.DataFrame(per_query)
pq["hit10"] = pq["rank"].apply(lambda r: r is not None and r <= 10)
print("\nRecall@10 по тому, где ответ в ролике:")
display(
    pq.pivot_table(
        index=["model", "chunking"], columns="where", values="hit10", aggfunc="mean"
    ).round(3)
)
print("\nRecall@10 по автору запроса:")
display(
    pq.pivot_table(
        index=["model", "chunking"], columns="author", values="hit10", aggfunc="mean"
    ).round(3)
)

In [ ]:
# Сохраняем результаты и транскрипты (пригодятся для MEM-30 и MEM-79) и скачиваем архив
import shutil
import zipfile

res.to_csv("comparison.csv", index=False, encoding="utf-8-sig")
pq.to_csv("per_query.csv", index=False, encoding="utf-8-sig")
shutil.make_archive("embeddings_check_results", "zip", ".", base_dir="transcripts")
with zipfile.ZipFile("embeddings_check_results.zip", "a") as z:
    z.write("comparison.csv")
    z.write("per_query.csv")
try:
    from google.colab import files

    files.download("embeddings_check_results.zip")
except Exception:
    print("Скачай embeddings_check_results.zip из панели файлов слева")